# PCOS-Informed Diabetes Classification in Women

**Authentic framing:** We classify **diabetes status** (`Diabetes_012`) among **women** in open CDC BRFSS 2015 health-indicator data. PCOS motivates the problem because of its established link to insulin resistance and Type 2 diabetes, but **this dataset has no PCOS diagnosis field**. We do not invent PCOS labels.

| Item | Value |
|------|-------|
| Raw source rows | 253,680 (before filtering) |
| Cohort | Female respondents (`Sex == 0`) |
| Target | 0 = no diabetes, 1 = prediabetes, 2 = diabetes |
| Paradigms | scikit-learn (multiple algorithms) + TensorFlow/Keras MLP with `tf.data` |


## 0. Environment

If you are on Colab, uncomment the clone/install cells as needed. Locally, install from `requirements.txt` and run this notebook from the repo root (or add the repo root to `sys.path`).


In [ ]:
from pathlib import Path
import sys

# Ensure project root is on the path whether the kernel cwd is repo root or notebooks/
CANDIDATES = [Path.cwd(), Path.cwd().parent]
ROOT = next((p for p in CANDIDATES if (p / "src").exists() and (p / "requirements.txt").exists()), Path.cwd())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("Project root:", ROOT)


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.config import CLASS_NAMES, DATA_RAW, RANDOM_STATE, TARGET_COL
from src.data_loading import load_raw_dataset, load_women_diabetes_frame
from src.preprocessing import engineer_features, feature_matrix_columns, stratified_splits
from src.traditional_ml import build_pipelines, tune_model, save_sklearn_model
from src.deep_learning import (
    build_mlp,
    compile_model,
    compute_class_weights,
    default_callbacks,
    make_tf_dataset,
    save_keras_model,
    set_tf_seed,
)
from src.evaluation import (
    metrics_table,
    plot_class_balance,
    plot_confusion,
    plot_learning_curve,
    plot_roc_ovr,
    summarize_classification,
)

pd.set_option("display.max_columns", 50)
print("Imports OK")


## 1b. Preprocessing audit, missingness, and BMI outliers

We document data quality before modeling: missingness by column, BMI plausible-range clipping, and an ablation later comparing engineered metabolic features. For full W&B offline logging of every run, prefer `python scripts/run_experiments.py` (sets `WANDB_MODE=offline` and writes `report/experiment_results/experiment_table.csv`).


In [ ]:
from src.preprocessing import diagnose_data_quality, handle_missing_and_outliers, iqr_outlier_counts

women_raw = load_women_diabetes_frame()
quality = diagnose_data_quality(women_raw)
print(quality.sort_values('missing_pct', ascending=False).head(10).to_string(index=False))
women_clean, clean_report = handle_missing_and_outliers(women_raw)
print('Cleaning report:', clean_report)
print('BMI IQR fences:', iqr_outlier_counts(women_clean['BMI']))
# Prefer cleaned frame in subsequent feature engineering
women = women_clean


## Experiment tracking (W&B offline)

```bash
export WANDB_MODE=offline
python scripts/run_experiments.py
# Inspect: report/experiment_results/experiment_table.csv and wandb/
```



## 1. Load raw data and verify the ≥20,000-row requirement

Place `diabetes_012_health_indicators_BRFSS2015.csv` in `data/raw/` (see `scripts/download_data.py`). **Do not commit the CSV to GitHub.**


In [ ]:
raw = load_raw_dataset()
print(f"Raw shape: {raw.shape}")
print(f"Raw rows >= 20,000? {len(raw) >= 20_000}")
raw.head()


In [ ]:
print("Sex value counts (0=female, 1=male in this cleaned file):")
print(raw["Sex"].value_counts(dropna=False))
print("\nDiabetes_012 distribution (full raw file):")
print(raw[TARGET_COL].value_counts(normalize=True).sort_index())


## 2. Define the women cohort

Filtering to women is a **cohort definition step**, not synthetic data generation. Sex is then dropped so it is not used as a model feature.


In [ ]:
women = load_women_diabetes_frame()
print(f"Women cohort shape: {women.shape}")
print(women[TARGET_COL].map(CLASS_NAMES).value_counts())
plot_class_balance(women[TARGET_COL], title="Diabetes class balance (women cohort)", filename="class_balance_women.png")
plt.show()


## 3. Feature engineering (metabolic phenotype proxies — not PCOS labels)

We add composites that reflect cardiometabolic burden discussed in PCOS–diabetes literature. Every engineered field is derived from existing BRFSS indicators.


In [ ]:
df = engineer_features(women)
feature_cols = feature_matrix_columns(df)
print(f"Using {len(feature_cols)} features:")
print(feature_cols)
df[["BMI", "HighBP", "HighChol", "MetabolicBurden", "LifestyleRisk", "BMI_x_HighBP", TARGET_COL]].describe()


## 4. Stratified train / validation / test splits


In [ ]:
X_train, X_val, X_test, y_train, y_val, y_test = stratified_splits(df)
# Keep only model features
X_train = X_train[feature_cols]
X_val = X_val[feature_cols]
X_test = X_test[feature_cols]
print(X_train.shape, X_val.shape, X_test.shape)
print("Train class mix:\n", y_train.value_counts(normalize=True).sort_index())


## 5. Traditional ML baseline suite

We train Logistic Regression, Random Forest, Gradient Boosting, and a calibrated Linear SVM. Class imbalance is addressed with class weights where supported; primary selection metric is **macro-F1**.


In [ ]:
pipelines = build_pipelines(feature_cols)
baseline_rows = []
fitted = {}

for name, pipe in pipelines.items():
    print(f"\n=== Fitting {name} ===")
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    proba = pipe.predict_proba(X_test) if hasattr(pipe, "predict_proba") else None
    summary = summarize_classification(y_test, pred, proba, title=f"{name} (baseline)")
    baseline_rows.append(summary)
    fitted[name] = pipe
    print({k: summary[k] for k in summary if k != "report"})

metrics_table(baseline_rows).sort_values("macro_f1", ascending=False)


## 6. Hyperparameter tuning (RandomizedSearchCV)

Tune the stronger baselines. Reduce `n_iter` if you need a quicker Colab run.


In [ ]:
# Choose models to tune (SVM calibration makes search slower; optional)
TO_TUNE = ["log_reg", "random_forest", "grad_boost"]
searches = {}
tuned_rows = []

# Fit tuning on train+val for a fairer final test comparison
X_tune = pd.concat([X_train, X_val], axis=0)
y_tune = pd.concat([y_train, y_val], axis=0)

for name in TO_TUNE:
    print(f"\n=== Tuning {name} ===")
    search = tune_model(name, build_pipelines(feature_cols)[name], X_tune, y_tune, n_iter=6, cv=3)
    searches[name] = search
    best = search.best_estimator_
    pred = best.predict(X_test)
    proba = best.predict_proba(X_test) if hasattr(best, "predict_proba") else None
    summary = summarize_classification(y_test, pred, proba, title=f"{name} (tuned)")
    tuned_rows.append(summary)
    fitted[f"{name}_tuned"] = best
    print("Best params:", search.best_params_)
    print({k: summary[k] for k in summary if k != "report"})

trad_table = metrics_table(baseline_rows + tuned_rows).sort_values("macro_f1", ascending=False)
trad_table


In [ ]:
# Select best traditional model by macro-F1 on the held-out test set
best_trad_name = trad_table.iloc[0]["title"].replace(" (baseline)", "").replace(" (tuned)", "")
# Map display title back to fitted dict key
cand_keys = [k for k in fitted if best_trad_name.split()[0] in k or best_trad_name in k]
# Prefer tuned variant when available
best_trad_key = next((k for k in [f"{best_trad_name}_tuned", best_trad_name] if k in fitted), list(fitted)[0])
# More reliable: pick by recomputing macro-F1
from sklearn.metrics import f1_score
best_trad_key, best_trad_f1 = None, -1.0
for k, model in fitted.items():
    f1 = f1_score(y_test, model.predict(X_test), average="macro", zero_division=0)
    if f1 > best_trad_f1:
        best_trad_key, best_trad_f1 = k, f1
best_trad = fitted[best_trad_key]
print(f"Best traditional model: {best_trad_key} | macro-F1={best_trad_f1:.4f}")

trad_pred = best_trad.predict(X_test)
trad_proba = best_trad.predict_proba(X_test)
plot_confusion(y_test, trad_pred, f"Confusion — {best_trad_key}", "cm_best_traditional.png")
plt.show()
plot_roc_ovr(y_test, trad_proba, f"ROC OvR — {best_trad_key}", "roc_best_traditional.png")
plt.show()


In [ ]:
# Learning curve on a fresh untuned RF pipeline for interpretability
lc_pipe = build_pipelines(feature_cols)["random_forest"]
plot_learning_curve(
    lc_pipe,
    X_tune,
    y_tune,
    title="Learning curve — Random Forest (macro-F1)",
    filename="learning_curve_rf.png",
    cv=3,
)
plt.show()


## 7. Deep learning with TensorFlow / Keras and `tf.data`

Functional-API MLP trained with class weights, early stopping, and ReduceLROnPlateau. Inputs are scaled using statistics from the training split only.


In [ ]:
from sklearn.preprocessing import StandardScaler

set_tf_seed(RANDOM_STATE)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_val_s = scaler.transform(X_val)
X_test_s = scaler.transform(X_test)

batch_size = 256
ds_train = make_tf_dataset(X_train_s, y_train.to_numpy(), batch_size=batch_size, shuffle=True)
ds_val = make_tf_dataset(X_val_s, y_val.to_numpy(), batch_size=batch_size, shuffle=False)
ds_test = make_tf_dataset(X_test_s, y_test.to_numpy(), batch_size=batch_size, shuffle=False)

class_weight = compute_class_weights(y_train.to_numpy())
print("Class weights:", class_weight)


In [ ]:
# Experiment A: wider network
configs = [
    {"name": "mlp_128_64_32", "hidden": (128, 64, 32), "dropout": 0.3, "lr": 1e-3},
    {"name": "mlp_64_32", "hidden": (64, 32), "dropout": 0.2, "lr": 1e-3},
    {"name": "mlp_256_128_64", "hidden": (256, 128, 64), "dropout": 0.4, "lr": 5e-4},
]

dl_histories = {}
dl_models = {}
dl_rows = []

for cfg in configs:
    print(f"\n=== Training {cfg['name']} ===")
    model = compile_model(
        build_mlp(n_features=X_train_s.shape[1], hidden_units=cfg["hidden"], dropout=cfg["dropout"]),
        learning_rate=cfg["lr"],
    )
    history = model.fit(
        ds_train,
        validation_data=ds_val,
        epochs=40,
        class_weight=class_weight,
        callbacks=default_callbacks(patience=7),
        verbose=1,
    )
    proba = model.predict(X_test_s, verbose=0)
    pred = proba.argmax(axis=1)
    summary = summarize_classification(y_test, pred, proba, title=cfg["name"])
    dl_rows.append(summary)
    dl_histories[cfg["name"]] = history
    dl_models[cfg["name"]] = model
    print({k: summary[k] for k in summary if k != "report"})

dl_table = metrics_table(dl_rows).sort_values("macro_f1", ascending=False)
dl_table


In [ ]:
best_dl_name = dl_table.iloc[0]["title"]
best_dl = dl_models[best_dl_name]
print("Best DL model:", best_dl_name)
best_dl.summary()


In [ ]:
# Training curves for the best DL model
hist = dl_histories[best_dl_name].history
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].plot(hist["loss"], label="train")
axes[0].plot(hist["val_loss"], label="val")
axes[0].set_title(f"Loss — {best_dl_name}")
axes[0].legend()
axes[1].plot(hist["accuracy"], label="train")
axes[1].plot(hist["val_accuracy"], label="val")
axes[1].set_title(f"Accuracy — {best_dl_name}")
axes[1].legend()
fig.tight_layout()
fig.savefig(ROOT / "figures" / "dl_training_curves.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
dl_proba = best_dl.predict(X_test_s, verbose=0)
dl_pred = dl_proba.argmax(axis=1)
plot_confusion(y_test, dl_pred, f"Confusion — {best_dl_name}", "cm_best_dl.png")
plt.show()
plot_roc_ovr(y_test, dl_proba, f"ROC OvR — {best_dl_name}", "roc_best_dl.png")
plt.show()


## 8. Head-to-head comparison and error analysis


In [ ]:
trad_summary = summarize_classification(y_test, trad_pred, trad_proba, title=f"Traditional: {best_trad_key}")
dl_summary = summarize_classification(y_test, dl_pred, dl_proba, title=f"Deep Learning: {best_dl_name}")
comparison = metrics_table([trad_summary, dl_summary])
comparison


In [ ]:
# Where does the best model fail? Focus on minority classes.
err = X_test.copy()
err["y_true"] = y_test.to_numpy()
err["y_pred"] = dl_pred
err["correct"] = err["y_true"] == err["y_pred"]
print("Error rate by true class:")
print(err.groupby("y_true")["correct"].apply(lambda s: 1 - s.mean()))
print("\nMean MetabolicBurden by true class (women cohort test set):")
print(err.groupby("y_true")["MetabolicBurden"].mean())


### Interpretation notes (edit after you run)

- Expect **class imbalance**: prediabetes is typically rare in BRFSS indicators; macro-F1 / OvR AUC matter more than raw accuracy.
- Tree ensembles often compete strongly on tabular BRFSS features; an MLP may match but not dominate without richer signals.
- Confusion between prediabetes and no-diabetes is clinically and statistically common.
- **Dataset limitation:** absence of PCOS labels means we cannot estimate PCOS-conditional diabetes risk directly—only women-level diabetes classification informed by that clinical context.


## 9. Save best model artifacts (for Google Drive; gitignored)


In [ ]:
trad_path = save_sklearn_model(best_trad, "best_traditional_ml.joblib")
# Persist scaler with DL model folder for reproducibility
import joblib
joblib.dump(scaler, ROOT / "models" / "dl_standard_scaler.joblib")
dl_path = save_keras_model(best_dl, "best_deep_learning.keras")
print("Saved:", trad_path)
print("Saved:", dl_path)
print("Upload these files + the zipped raw CSV to your Google Drive deliverable folder.")


## 10. Reproducibility checklist

1. Raw CSV stays in `data/raw/` and Google Drive only — **not** on GitHub.
2. Record your exact package versions (`pip freeze`) in the report appendix.
3. Replace README / report `TODO` links with GitHub, Drive, and video URLs.
4. In the written report, keep the authenticity statement near the start.
